<a href="https://colab.research.google.com/github/lawrence-kagugo/kenya-telecom-churn-analysis/blob/main/notebooks/03_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kenya Telecom Customer Churn — Data Cleaning
**Step:** 7 - Data Cleaning

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/lawrence-kagugo/kenya-telecom-churn-analysis/refs/heads/main/data/raw/kenya_telecom_customer_churn_dataset.csv"
df = pd.read_csv(url)
print("Loaded:", df.shape)

Loaded: (5000, 70)


In [3]:
# Convert the three date columns from text (object) to actual datetime type.
# errors='coerce' means: if any value can't be parsed as a date, turn it into
# NaT (Not a Time) instead of crashing - safer for messy real-world data.
# We expect ChurnDate to have NaT for all retained customers (structural, not an error).
df['DateOfBirth'] = pd.to_datetime(df['DateOfBirth'], errors='coerce')
df['JoinDate'] = pd.to_datetime(df['JoinDate'], errors='coerce')
df['ChurnDate'] = pd.to_datetime(df['ChurnDate'], errors='coerce')

# Confirm the fix worked - dtype should now show datetime64 instead of object
print(df[['DateOfBirth', 'JoinDate', 'ChurnDate']].dtypes)

DateOfBirth    datetime64[ns]
JoinDate       datetime64[ns]
ChurnDate      datetime64[ns]
dtype: object


In [4]:
# If coerce turned any unparseable date into NaT, we'd now have MORE missing
# values than before. Compare against what we already know from Step 5:
# ChurnDate should still show exactly 3928 missing (matching retained customers).
# DateOfBirth and JoinDate should show 0 missing (they had none before).
print("DateOfBirth missing:", df['DateOfBirth'].isnull().sum())
print("JoinDate missing:", df['JoinDate'].isnull().sum())
print("ChurnDate missing:", df['ChurnDate'].isnull().sum())

DateOfBirth missing: 0
JoinDate missing: 0
ChurnDate missing: 3928


## Cleaning Action 1: Date column conversion
Converted DateOfBirth, JoinDate, and ChurnDate from object (text) to datetime64.
Verified no values were lost in conversion (missing counts unchanged:
DateOfBirth=0, JoinDate=0, ChurnDate=3928 matching retained customer count).

In [5]:
# Re-check the ~1-2% missing columns, this time noting their data type,
# since numeric vs categorical missing values need different treatment.
cols_to_review = ['ProductRating', 'Occupation', 'EducationLevel', 'CommunityParticipation',
                   'AveragePurchaseValue', 'ProfitMarginPercent', 'ReferralSource',
                   'MarketingClickRate', 'PreferredCommunication', 'SurveyScore',
                   'AverageSessionMinutes', 'EmailOpenRate', 'AverageResolutionHours']

for col in cols_to_review:
    print(f"{col}: dtype={df[col].dtype}, missing={df[col].isnull().sum()}")

ProductRating: dtype=float64, missing=92
Occupation: dtype=object, missing=85
EducationLevel: dtype=object, missing=83
CommunityParticipation: dtype=object, missing=77
AveragePurchaseValue: dtype=float64, missing=75
ProfitMarginPercent: dtype=float64, missing=75
ReferralSource: dtype=object, missing=75
MarketingClickRate: dtype=float64, missing=74
PreferredCommunication: dtype=object, missing=67
SurveyScore: dtype=float64, missing=65
AverageSessionMinutes: dtype=float64, missing=64
EmailOpenRate: dtype=float64, missing=62
AverageResolutionHours: dtype=float64, missing=59


In [6]:
# For each numeric column with missing values, compare mean vs median.
# If they're close, distribution is roughly symmetric (mean is safe).
# If they're far apart, distribution is skewed (median is safer).
numeric_cols = ['ProductRating', 'AveragePurchaseValue', 'ProfitMarginPercent',
                 'MarketingClickRate', 'SurveyScore', 'AverageSessionMinutes',
                 'EmailOpenRate', 'AverageResolutionHours']

for col in numeric_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    print(f"{col}: mean={mean_val:.2f}, median={median_val:.2f}, diff={abs(mean_val-median_val):.2f}")

ProductRating: mean=4.08, median=4.00, diff=0.08
AveragePurchaseValue: mean=1491.01, median=1487.04, diff=3.97
ProfitMarginPercent: mean=17.92, median=18.00, diff=0.08
MarketingClickRate: mean=11.05, median=9.60, diff=1.45
SurveyScore: mean=81.44, median=82.00, diff=0.56
AverageSessionMinutes: mean=24.94, median=24.90, diff=0.04
EmailOpenRate: mean=39.97, median=38.55, diff=1.42
AverageResolutionHours: mean=12.02, median=10.00, diff=2.02


In [7]:
# Use median for the two skewed columns (robust to skew), mean for the rest
# (roughly symmetric, so mean is a fair representative value).
skewed_cols = ['MarketingClickRate', 'AverageResolutionHours']
symmetric_cols = ['ProductRating', 'AveragePurchaseValue', 'ProfitMarginPercent',
                   'SurveyScore', 'AverageSessionMinutes', 'EmailOpenRate']

for col in skewed_cols:
    df[col] = df[col].fillna(df[col].median())

for col in symmetric_cols:
    df[col] = df[col].fillna(df[col].mean())

# Confirm no missing values remain in these 8 columns
print(df[skewed_cols + symmetric_cols].isnull().sum())

MarketingClickRate        0
AverageResolutionHours    0
ProductRating             0
AveragePurchaseValue      0
ProfitMarginPercent       0
SurveyScore               0
AverageSessionMinutes     0
EmailOpenRate             0
dtype: int64


In [8]:
# Fill missing categorical values with an explicit "Unknown" label rather
# than guessing via mode - preserves the honesty that this data point
# was genuinely not collected, rather than fabricating a value.
categorical_missing_cols = ['Occupation', 'EducationLevel', 'CommunityParticipation',
                              'ReferralSource', 'PreferredCommunication']

for col in categorical_missing_cols:
    df[col] = df[col].fillna('Unknown')

# Confirm no missing values remain
print(df[categorical_missing_cols].isnull().sum())

Occupation                0
EducationLevel            0
CommunityParticipation    0
ReferralSource            0
PreferredCommunication    0
dtype: int64


In [9]:
# Final check: confirm the entire dataset is now missing-free EXCEPT
# ChurnDate/ChurnReason, which should still show 3928 missing (structural,
# intentionally left as-is since it correctly reflects "not applicable").
missing_final = df.isnull().sum()
missing_final = missing_final[missing_final > 0]
print(missing_final)

ChurnDate      3928
ChurnReason    3928
dtype: int64


## Cleaning Action 2: Missing value imputation
- 8 numeric columns (1-2% missing each): imputed using median for skewed
  columns (MarketingClickRate, AverageResolutionHours) and mean for roughly
  symmetric columns (ProductRating, AveragePurchaseValue, ProfitMarginPercent,
  SurveyScore, AverageSessionMinutes, EmailOpenRate). Chose based on relative
  mean-median gap, not raw difference, since columns have very different scales.
- 5 categorical columns (1-2% missing each): imputed with explicit "Unknown"
  label rather than mode, to avoid fabricating values with no structural
  justification for the missingness.
- ChurnDate/ChurnReason (78.56% missing): left as-is - structural missingness,
  correctly reflects customers who never churned.

Final check confirms only ChurnDate/ChurnReason remain missing (3928 each,
matching retained customer count).

In [10]:
# Save the cleaned dataset to data/processed/ - keeping the raw file
# in data/raw/ completely untouched, per our reproducibility convention.
df.to_csv('kenya_telecom_churn_cleaned.csv', index=False)
print("Saved locally. Now upload this file to data/processed/ in your GitHub repo.")

Saved locally. Now upload this file to data/processed/ in your GitHub repo.
